# ANPR training on Colab A100 (skeleton)
Training only. Reproduce each run from: git commit + dataset hash + config yaml + seed.
Before running: add the Colab secret `GITHUB_TOKEN` (read-only access to the repository) and upload `dataset_v5.zip` to the Drive folder. Steps: `docs/setup_android_colab.md`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, json, time, platform
DRIVE = '/content/drive/MyDrive/anpr'
REPO = 'parth20jakl/Final_Year_Project'
RUN_ID = time.strftime('%Y%m%d_%H%M%S')
os.makedirs(f'{DRIVE}/runs/{RUN_ID}', exist_ok=True)

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# private repository: the token comes from Colab Secrets and is never printed or saved in the clone
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
if not os.path.exists('/content/repo/.git'):
    r = subprocess.run(['git', 'clone', f'https://{token}@github.com/{REPO}.git', '/content/repo'],
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stderr.replace(token, '***')
    subprocess.run(['git', '-C', '/content/repo', 'remote', 'set-url', 'origin',
                    f'https://github.com/{REPO}.git'], check=True)
%cd /content/repo
!pip install -q -r colab/requirements-colab.txt
!git rev-parse HEAD

In [ ]:
# Dataset: download dataset_v5.zip from the Drive folder (works if the folder is yours OR shared with this Colab account)
import hashlib, io, os
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
FOLDER_ID = '1CmwJB6IxwyJi0SQSC6JXJp4KiEar0Dgj'          # folder that holds dataset_v5.zip
EXPECTED_SHA256 = 'c274dea4453bd10e0664603a97b1a0ad54a7804cd825cc50810ea46b16fa08ba'
drive_api = build('drive', 'v3')
res = drive_api.files().list(q=f"'{FOLDER_ID}' in parents and name='dataset_v5.zip' and trashed=false",
                             fields='files(id,name,size)', supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
assert res['files'], 'dataset_v5.zip not visible: share the Drive folder with this Colab account (Viewer) and rerun'
fid = res['files'][0]['id']
os.makedirs('/content/data', exist_ok=True)
with open('/content/dataset_v5.zip', 'wb') as f:
    dl = MediaIoBaseDownload(f, drive_api.files().get_media(fileId=fid, supportsAllDrives=True), chunksize=64*1024*1024)
    done = False
    while not done:
        status, done = dl.next_chunk()
        print(f'{int(status.progress()*100)}%', end=' ')
h = hashlib.sha256()
with open('/content/dataset_v5.zip', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
assert h.hexdigest() == EXPECTED_SHA256, f'checksum mismatch: {h.hexdigest()}'
print()
print('checksum OK')
!unzip -q -o /content/dataset_v5.zip -d /content/data
!cat /content/data/dataset_v5/version.json

In [ ]:
# environment record (mandatory per run)
import importlib.metadata as md
env = {'run_id': RUN_ID, 'python': platform.python_version(), 'torch': torch.__version__,
       'gpu': torch.cuda.get_device_name(0), 'commit': subprocess.getoutput('git rev-parse HEAD'),
       'packages': {d.metadata['Name']: d.version for d in md.distributions()}}
json.dump(env, open(f'{DRIVE}/runs/{RUN_ID}/env.json', 'w'), indent=1)

In [ ]:
# example: baseline training with resume. Hyper-parameters come from a yaml in configs/, not from here.
from ultralytics import YOLO
# model = YOLO('yolo11n.pt')
# model.train(data='/content/data/dataset_v5/anpr_detect.yaml', imgsz=640, epochs=100, batch=32, seed=42,
#             project=f'{DRIVE}/runs', name=RUN_ID, save_period=1, exist_ok=True)
# resume after disconnect: YOLO(f'{DRIVE}/runs/{RUN_ID}/weights/last.pt').train(resume=True)